In [11]:
import requests
import io
from bs4 import BeautifulSoup as bs
import pandas as pd
import re

In [12]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

1. Start by using either the inspector or by viewing the page source. Can you identify a tag that might be helpful for finding the names of all performers? For now, just worry about the headliner and don't worry about the opener. (Eg. For Vince Gill, featuring Wendy Moten, we only care about Vince Gill.) Make use of this to create a list containing just the names of each inductee.

In [13]:
response = requests.get('https://www.ryman.com/events/', headers=headers)

In [14]:
response.status_code

200

In [15]:
soup = bs(response.text)

In [16]:
performers = [x.get_text() for x in soup.find_all('a', id=re.compile(r'event_listing_title'))]

2. Next, try and find a tag that could be used to find the date and time for each show. Extract these into a list. Challenge: Convert these into two lists, one containing the date and the other containing the time. (Eg. split Mar 9, 2023 8:00 PM into Mar 9, 2023 and 8:00 PM.)

In [132]:
datetime = [x.get_text().strip() for x in soup.find_all('div', class_='date') if x.get_text()]

In [54]:
date = [dt[:-8] if dt[-1] == 'M' else dt for dt in datetime]

In [57]:
date

['Apr 13, 2026',
 'Apr 14, 2026',
 'Apr 15, 2026',
 'Apr 16, 2026',
 'Apr 17, 2026',
 'Apr 18, 2026',
 'Apr 19, 2026',
 'Apr 21, 2026',
 'Apr 21, 2026',
 'Apr 25, 27 & 28',
 'Apr 26, 2026',
 'Apr 29, 2026']

In [59]:
time = [dt[-7:] if dt[-1] == 'M' else '-' for dt in datetime]

In [60]:
time

['7:00 PM',
 '7:00 PM',
 '7:30 PM',
 '7:00 PM',
 '8:00 PM',
 '8:00 PM',
 '7:00 PM',
 '5:00 PM',
 '7:00 PM',
 '-',
 '7:30 PM',
 '5:30 PM']

3. Take the lists you created on parts 1 and 2 and convert them into a pandas DataFrame.

In [62]:
ryman_events = pd.DataFrame({'performers': performers, 'date': date, 'time':time})

In [63]:
ryman_events

,performers,date,time
0,Kid Rock's Comedy Jam,"Apr 13, 2026",7:00 PM
1,Shxtsngigs,"Apr 14, 2026",7:00 PM
2,Theo Von,"Apr 15, 2026",7:30 PM
3,Heather McMahan,"Apr 16, 2026",7:00 PM
4,David Spade,"Apr 17, 2026",8:00 PM
5,Joey Diaz,"Apr 18, 2026",8:00 PM
6,Nate Jackson,"Apr 19, 2026",7:00 PM
7,Ryman Sidewalk Sessions with Hair Force One,"Apr 21, 2026",5:00 PM
8,Styx,"Apr 21, 2026",7:00 PM
9,Hayley Williams At A Bachelorette Party,"Apr 25, 27 & 28",-


4. Bonus #1: Add to your data frame the opening act for all shows that list an opener.

In [154]:
[re.findall(r'(with|featuring)\s(.*)', x.get_text().strip())[0][1] for x in soup.find_all('h4', class_='tagline') if re.search(r'(with|featuring)', x.get_text())]

['Water From Your Eyes']

In [206]:
len(soup.find_all('div', class_='eventItem', attrs={'aria-label':'event'}))

12

In [177]:
[x.find_all('h4', class_='tagline')[0].get_text().strip() if x.find_all('h4', class_='tagline') else '-' for x in soup.find_all('div', class_='eventItem')]

['-',
 '-',
 '-',
 '-',
 '-',
 'Ages 16 & Up',
 '-',
 '-',
 'Free & Open to the Public',
 '-',
 'with Water From Your Eyes',
 '-',
 'Free & Open to the Public']

In [208]:
openers = []
for x in soup.find_all('div', class_='eventItem', attrs={'aria-label':'event'}):
    div = x.find('h4', class_='tagline')
    if div:
        if re.search(r'(with|featuring)', div.get_text()):
            opener = re.findall(r'(with|featuring)\s(.*)', div.get_text().strip())[0][1]
        else:
            opener = '-'
    else:
        opener = '-'
    openers.append(opener)
openers

['-', '-', '-', '-', '-', '-', '-', '-', '-', 'Water From Your Eyes', '-', '-']

In [209]:
ryman_events['openers'] = openers

In [210]:
ryman_events

,performers,date,time,openers
0,Kid Rock's Comedy Jam,"Apr 13, 2026",7:00 PM,-
1,Shxtsngigs,"Apr 14, 2026",7:00 PM,-
2,Theo Von,"Apr 15, 2026",7:30 PM,-
3,Heather McMahan,"Apr 16, 2026",7:00 PM,-
4,David Spade,"Apr 17, 2026",8:00 PM,-
5,Joey Diaz,"Apr 18, 2026",8:00 PM,-
6,Nate Jackson,"Apr 19, 2026",7:00 PM,-
7,Ryman Sidewalk Sessions with Hair Force One,"Apr 21, 2026",5:00 PM,-
8,Styx,"Apr 21, 2026",7:00 PM,-
9,Hayley Williams At A Bachelorette Party,"Apr 25, 27 & 28",-,Water From Your Eyes


5. Bonus #2: Now, let's see if we can get the results beyond the first page. For this, you'll need to Web Developer Tools of your browser and navigate to the Network tab. Click the "Load More Events" button and you should see a GET request to the www.ryman.com domain.<br>a. Inspect this request and you should see that it goes to a URL like "https://www.ryman.com/events/events_ajax/24?category=0&venue=0&team=0&exclude=&per_page=12&came_from_page=event-list-page". In your Jupyter notebook, send a get request to this url and inspect the results.<br>b. You should find that the results that you get are HTML, but that they are not exactly formatted in a way that can be parsed. See if you can clean up the results set so that you can extract out the same information as above.<br>c. Create a DataFrame that contains data for the next 60 shows.

In [123]:
response2 = requests.get('https://www.ryman.com/events/events_ajax/12?category=0&venue=0&team=0&exclude=&per_page=12&came_from_page=event-list-page', headers=headers)

In [124]:
soup2 = bs(response2.text.replace('\\n', '').replace('\\t', '').replace('\\', ''))

In [125]:
# print(response2.text.replace('\\n', '').replace('\\t', '').replace('\\', ''))

In [126]:
# print(soup2.prettify())

In [127]:
[x.get_text() for x in soup2.find_all('a', id=re.compile(r'event_listing_title'))]

['Mac DeMarco',
 'Nitty Gritty Dirt Band',
 'Herbie Hancock',
 'Black Label Society',
 'Ryman Sidewalk Sessions with Rebecca Lee Daniels',
 'Drew & Ellie Holcomb',
 'Treaty Oak Revival',
 'Ryman Sidewalk Sessions with Brassfield',
 'Trace Adkins',
 'Chicago',
 'America: The Happy Trails Tour 2026',
 'Ryman Sidewalk Sessions with Brothers Revolt']

In [131]:
[x.get_text().strip() for x in soup2.find_all('div', class_='date') if x.get_text()]

['May 12, 2026 7:30 PM',
 'May 13, 2026 7:30 PM',
 'May 13, 2026 8:00 PM',
 'May 14, 2026 7:30 PM',
 'May 15, 2026 6:00 PM',
 'May 15-16, 2026',
 'May 17, 2026 7:30 PM',
 'May 22, 2026 6:00 PM',
 'May 22-23, 2026',
 'May 27, 2026 7:30 PM',
 'May 28, 2026 7:00 PM',
 'May 29, 2026 5:30 PM']

In [229]:
url_start='https://www.ryman.com/events/events_ajax/'
url_end='?category=0&venue=0&team=0&exclude=&per_page=12&came_from_page=event-list-page'
next_ryman_events = pd.DataFrame()
for i in range(5):
    url = url_start + str((i+1) * 12) + url_end
    response_ryman = requests.get(url, headers=headers)
    soup_ryman = bs(response_ryman.text.replace('\\n', '').replace('\\t', '').replace('\\', ''))
    performers_ryman = [x.get_text() for x in soup_ryman.find_all('a', id=re.compile(r'event_listing_title'))]
    openers_ryman = []
    for x in soup_ryman.find_all('div', class_='eventItem', attrs={'aria-label':'event'}):
        div = x.find('h4', class_='tagline')
        if div:
            if re.search(r'(with|featuring)', div.get_text()):
                opener = re.findall(r'(with|featuring)\s(special\sguest)?s?(.*)', div.get_text().strip())[0][2]
            else:
                opener = '-'
        else:
            opener = '-'
        if opener == '':
            opener = '-'
        openers_ryman.append(opener)
    datetime_ryman = [x.get_text().strip() for x in soup_ryman.find_all('div', class_='date') if x.get_text()]
    date_ryman = [dt[:-8] if dt[-1] == 'M' else dt for dt in datetime_ryman]
    time_ryman = [dt[-7:] if dt[-1] == 'M' else '-' for dt in datetime_ryman]
    next_ryman_events = pd.concat([next_ryman_events, pd.DataFrame({'performer': performers_ryman, 'opener': openers_ryman,'date': date_ryman, 'time':time_ryman})]).reset_index(drop=True)
next_ryman_events

,performer,opener,date,time
0,RAYE,Absolutely and AMMA,"Apr 29, 2026",7:30 PM
1,Chelsea Handler,-,"Apr 30, 2026",8:00 PM
2,Ani DiFranco,Valerie June,"May 1, 2026",8:00 PM
3,An Evening with Josh Ritter,-,"May 2, 2026",8:00 PM
4,Tracy Lawrence,-,"May 3, 2026",7:30 PM
5,Fender Presents: Tele Town,"Brad Paisley, Brent Mason, Brothers Osborne, D...","May 4, 2026",7:30 PM
6,Courtney Barnett,Truman Sinclair,"May 5, 2026",7:30 PM
7,Ryman Sidewalk Sessions with Matt Mann & the S...,-,"May 6, 2026",5:30 PM
8,A Night With David Lee Roth,-,"May 6, 2026",7:30 PM
9,SatchVai Band Ft Joe Satriani & Steve Vai,Animals As Leaders,"May 7, 2026",8:00 PM
